# Buổi 4 · Perceptron phân loại phân khúc nhà

**Case study thực tế:** từ thông tin vị trí và đặc điểm căn nhà, xây một node nơ-ron cơ bản để dự đoán căn nhà thuộc nhóm **giá/m² cao** hay **phổ thông**. Node này có thể được ghép với các node khác trong một mạng nơ-ron lớn hơn.

## Tình huống

Một nhóm phân tích cần sàng lọc nhanh danh sách nhà trước khi chuyên viên xem chi tiết. Ta dùng ngưỡng **85 triệu đồng/m²** để tạo nhãn học:

- `0` — Phổ thông: giá/m² dưới 85 triệu đồng.
- `1` — Giá/m² cao: giá/m² từ 85 triệu đồng trở lên.

Sau bài này, bạn có thể: chuẩn bị feature số và phân loại; chia train/test; huấn luyện Perceptron; đọc confusion matrix; giải thích `net = wᵀx + b`; và nhận ra data leakage.

> Dữ liệu trong repo là dữ liệu mô phỏng phục vụ học tập, không phải dữ liệu định giá thị trường.

In [ ]:
from importlib.util import module_from_spec, spec_from_file_location
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

ROOT = Path.cwd()
if not (ROOT / 'data' / 'sample_houses.csv').exists():
    ROOT = ROOT.parent

script_path = ROOT / 'bai-tap' / 'buoi4_perceptron_phan_khuc_nha.py'
spec = spec_from_file_location('buoi4_housing', script_path)
case = module_from_spec(spec)
sys.modules[spec.name] = case
assert spec.loader is not None
spec.loader.exec_module(case)

pd.set_option('display.max_columns', 20)

## 1. Đọc dữ liệu và tạo nhãn

CSV đã nằm trong `data/sample_houses.csv`, nên notebook chạy được khi không có Internet. Hãy đọc 5 dòng đầu và kiểm tra ý nghĩa từng cột trước khi huấn luyện.

In [ ]:
THRESHOLD = 85.0
data = case.load_housing_data(ROOT / 'data' / 'sample_houses.csv', threshold=THRESHOLD)
print(f'Số dòng: {len(data)} | Số cột: {data.shape[1]}')
data.head()

In [ ]:
class_counts = (
    data[case.TARGET_COLUMN]
    .map(case.CLASS_NAMES)
    .value_counts()
    .rename_axis('nhãn')
    .to_frame('số mẫu')
)
class_counts['tỷ lệ'] = (class_counts['số mẫu'] / len(data)).round(3)
class_counts

### Điểm dừng · Vì sao không dùng cột giá?

Nhãn được tạo trực tiếp từ `price_per_m2_million_vnd`. Nếu đưa cột này hoặc `price_million_vnd` vào đầu vào, mô hình gần như được nhìn thấy đáp án. Đây là **data leakage**: điểm số đẹp nhưng không phản ánh khả năng suy luận từ các đặc điểm căn nhà.

Trước khi chạy ô tiếp theo, hãy ghi lại: nếu lỡ thêm giá/m² vào feature, accuracy sẽ có xu hướng tăng hay giảm? Vì sao?

In [ ]:
assert not set(case.LEAKAGE_COLUMNS) & set(case.FEATURE_COLUMNS)
x_train, x_test, y_train, y_test = case.split_dataset(
    data, test_size=0.25, random_state=42
)
print('Feature:', case.FEATURE_COLUMNS)
print('Train:', x_train.shape, '| Test:', x_test.shape)
print('Tỷ lệ lớp cao trong train/test:', round(y_train.mean(), 3), round(y_test.mean(), 3))

## 2. Huấn luyện một node Perceptron

Pipeline thực hiện ba việc: chuẩn hoá feature số, one-hot encoding feature phân loại, rồi đưa vector kết quả vào một Perceptron. Node tính `net = wᵀx + b`; nếu `net ≥ 0` thì dự đoán lớp 1. Đây vẫn là node cơ bản đã học — chỉ có nhiều feature thực tế hơn.

In [ ]:
model = case.build_model(random_state=42, eta0=0.1)
model.fit(x_train, y_train)
predictions = model.predict(x_test)
net_scores = model.decision_function(x_test)

print('Đã huấn luyện xong.')
print('5 giá trị net đầu tiên:', net_scores[:5].round(3))
print('5 nhãn dự đoán đầu tiên:', predictions[:5])

## 3. Đánh giá trên dữ liệu chưa thấy

Không chỉ nhìn accuracy. Với lớp `Giá/m² cao`, **precision** trả lời “các căn bị gắn nhãn cao đúng bao nhiêu?”, còn **recall** trả lời “mô hình tìm thấy bao nhiêu căn thật sự thuộc lớp cao?”.

In [ ]:
report = pd.DataFrame(
    classification_report(
        y_test,
        predictions,
        labels=[0, 1],
        target_names=[case.CLASS_NAMES[0], case.CLASS_NAMES[1]],
        output_dict=True,
        zero_division=0,
    )
).transpose()
report.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(5.8, 4.5))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    predictions,
    labels=[0, 1],
    display_labels=[case.CLASS_NAMES[0], case.CLASS_NAMES[1]],
    cmap='Blues',
    colorbar=False,
    ax=ax,
)
ax.set_title('Perceptron · tập test')
plt.tight_layout()
plt.show()

### Câu hỏi phân tích

1. Ghi số `TN`, `FP`, `FN`, `TP` từ ma trận.
2. Trong tình huống sàng lọc nhà, `FP` hay `FN` gây tốn thời gian hơn?
3. Accuracy có đủ để kết luận mô hình tốt không? Liên hệ với tỷ lệ hai lớp.

## 4. Nối trọng số với công thức trên lớp

Trọng số dương kéo `net` về phía lớp `Giá/m² cao`; trọng số âm kéo về phía `Phổ thông`. Đây là **mối liên hệ trong bộ dữ liệu mô phỏng**, không phải kết luận nhân quả về thị trường.

In [ ]:
weights = case.feature_weight_table(model)
print('5 trọng số dương lớn nhất:')
display(weights.head(5))
print('5 trọng số âm lớn nhất về độ lớn:')
display(weights.tail(5).sort_values('weight'))
print('Bias b =', round(float(model.named_steps['perceptron'].intercept_[0]), 3))

In [ ]:
errors = x_test.copy()
errors['giá/m² thật'] = data.loc[x_test.index, case.TARGET_SOURCE_COLUMN]
errors['nhãn thật'] = y_test.to_numpy()
errors['dự đoán'] = predictions
errors['net'] = net_scores.round(3)
errors = errors[errors['nhãn thật'] != errors['dự đoán']]
print(f'Số mẫu dự đoán sai: {len(errors)}')
errors.sort_values('net')

## 5. Chạy thử trên căn nhà mới

Hãy dự đoán bằng trực giác trước khi chạy. Chú ý `net` cho biết mẫu nằm phía nào của ranh giới; trị tuyệt đối càng gần 0 thì mẫu càng gần ranh giới quyết định.

In [ ]:
new_houses = [
    {
        'city': 'Ho Chi Minh City', 'district': 'Thu Duc',
        'building_type': 'apartment', 'area_m2': 70.0,
        'bedrooms': 2, 'bathrooms': 2, 'floors': 1, 'year_built': 2020,
    },
    {
        'city': 'Ha Noi', 'district': 'Hoan Kiem',
        'building_type': 'townhouse', 'area_m2': 75.0,
        'bedrooms': 3, 'bathrooms': 2, 'floors': 4, 'year_built': 2018,
    },
]
case.predict_new_houses(model, new_houses)

## 6. Bài tập nộp

### Phần A · Bắt buộc

1. Chạy toàn bộ notebook và báo cáo accuracy, precision, recall, F1 của lớp cao.
2. Chọn hai mẫu dự đoán sai; giải thích bằng `net`, nhãn thật và đặc điểm đầu vào.
3. Tự tạo một căn nhà mới hợp lệ, ghi dự đoán bằng tay rồi so với mô hình.
4. Giải thích trong 3–5 câu vì sao hai cột giá không được làm feature.

### Phần B · Thử nghiệm

Chạy lại với `threshold = 80` và `threshold = 90`. Lập bảng gồm: ngưỡng, số mẫu mỗi lớp, accuracy, precision, recall và F1. Giải thích vì sao đổi cách tạo nhãn có thể làm đổi kết quả.

### Phần C · Nâng cao

Bỏ `StandardScaler` rồi huấn luyện lại cùng seed. So sánh kết quả và số epoch (`n_iter_`). Từ đó giải thích vì sao chuẩn hoá quan trọng khi `area_m2`, `year_built` và số phòng có thang đo rất khác nhau.

In [ ]:
# Khung chạy Phần B — thêm các chỉ số của bạn vào bảng kết quả.
experiment_rows = []
for threshold in [80.0, 85.0, 90.0]:
    experiment_data, experiment = case.run_experiment(
        threshold=threshold, test_size=0.25, random_state=42, eta0=0.1
    )
    counts = experiment_data[case.TARGET_COLUMN].value_counts()
    experiment_rows.append({
        'ngưỡng': threshold,
        'phổ thông': int(counts.get(0, 0)),
        'giá/m² cao': int(counts.get(1, 0)),
        **{name: round(value, 3) for name, value in experiment.metrics.items()},
    })
pd.DataFrame(experiment_rows)

## Checklist trước khi nộp

- [ ] Restart kernel và Run All không lỗi.
- [ ] Có câu trả lời cho ba câu hỏi confusion matrix.
- [ ] Có phân tích hai mẫu sai, không chỉ chép chỉ số.
- [ ] Có một mẫu nhà mới do chính bạn tạo.
- [ ] Có bảng so sánh ba ngưỡng.
- [ ] Không thêm hai cột giá vào feature.